Gradient Boosting Models:
* Gradient Boosting Classifier (xgboost, lightgbm)
* Use StratifiedKFold for better train/test split
* Confirm results using confusion matrix, recall, precision, F1 scores

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report
from sklearn.preprocessing import OrdinalEncoder

In [36]:
# use cvd risk level not cvd risk score
# blood pressure categories found from https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings
# remove systolic, diastolic, and blood pressure, just keep blood pressure category (determined by systolic over diastolic values)

df = pd.read_csv('cvd_dataset.csv')
df_edited = df.drop(columns=['Blood Pressure (mmHg)','Systolic BP','Diastolic BP'])

# ordinal encoding categories
sex_categories = ['F','M'] 
physical_activity_categories = ['Low','Moderate','High']
blood_pressure_categories = ['Normal','Elevated','Hypertension Stage 1','Hypertension Stage 2']
# smoking, diabetes, family history all Y/N (0,1)
yn_categories = ['Y','N']
# cvd risk levels (0,1,2)
risk_categories = ['LOW','INTERMEDIARY','HIGH']

encoder = OrdinalEncoder(categories=[sex_categories, physical_activity_categories, blood_pressure_categories, yn_categories, yn_categories, yn_categories,risk_categories])
df_edited[['Sex','Physical Activity Level','Blood Pressure Category','Smoking Status','Diabetes Status','Family History of CVD','CVD Risk Level']] = encoder.fit_transform(df_edited[['Sex','Physical Activity Level','Blood Pressure Category','Smoking Status','Diabetes Status','Family History of CVD','CVD Risk Level']])

X = df_edited.drop(columns=['CVD Risk Level','CVD Risk Score'])
y = df_edited['CVD Risk Level']

In [ ]:
# https://www.geeksforgeeks.org/machine-learning/stratified-k-fold-cross-validation/
# gradient boosting machines
# builds decision trees in a serial fashion -- each tree is fit to the gradient of the loss function of the current stage's ensemble

kf = StratifiedKFold(n_splits=50,shuffle=True,random_state=42)
metrics = []
all_y_true = []
all_y_pred = []

cust_params = {'objective':'multiclass',
          'metric':'multi_logloss',
          'boosting_type':'gbdt',
          'num_leaves':31,
          'max_depth':-1, # higher depth leads to overfitting
          'n_estimators':100,
          'learning_rate':0.03,
          'min_data_in_leaf':2,
          'random_state':0,
          'num_class':3,
          'feature_fraction':1,
          'bagging_fraction':1,
          'verbosity':-1}

fold_num = 1
for train_index, test_index in kf.split(X,y): # splits dataset into stratified train-test indices
    X_train, X_test = X.iloc[train_index], X.iloc[test_index] # features
    y_train, y_test = y[train_index], y[test_index] # labels
    
    # creation of lightgbm datasets
    train_lgb_data = lgb.Dataset(X_train, label=y_train)
    test_lgb_data = lgb.Dataset(X_test, label=y_test)

    lgb_model = lgb.train(params=cust_params, train_set=train_lgb_data, num_boost_round=100,valid_sets=test_lgb_data)
    
    y_pred = lgb_model.predict(X_test)
    y_pred_labels = np.argmax(y_pred, axis=1)

    accuracy = accuracy_score(y_test, y_pred_labels)
    print(fold_num, ': ',accuracy)
    fold_num += 1
    metrics.append(accuracy)
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

print(classification_report(all_y_true, all_y_pred, target_names=encoder.categories_[0]))
cm = confusion_matrix(all_y_true, all_y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (All Folds)')
plt.tight_layout()
plt.show()
# # cross_val_results = cross_val_score(model, X, y_encoded, cv=kf, scoring ='accuracy')

print("Cross-Validation Results (Accuracy):")
for i, result in enumerate(metrics, 1):
    print(f"  Fold {i}: {result * 100:.2f}%")

TypeError: Cannot use Dataset instance for prediction, please use raw data instead

In [48]:
y_pred

array([[0.0505123 , 0.58182373, 0.36766398],
       [0.15206538, 0.21948847, 0.62844615],
       [0.05298846, 0.43239847, 0.51461307],
       [0.02441372, 0.26894308, 0.7066432 ],
       [0.15529012, 0.60968657, 0.23502331],
       [0.15927662, 0.53811338, 0.30261   ],
       [0.03488558, 0.41411203, 0.55100239],
       [0.3834165 , 0.56604285, 0.05054065],
       [0.07472156, 0.57329411, 0.35198433],
       [0.11544983, 0.56668282, 0.31786735],
       [0.01586041, 0.12291829, 0.8612213 ],
       [0.05679563, 0.42839216, 0.51481221],
       [0.04177616, 0.27782337, 0.68040047],
       [0.12218708, 0.45819673, 0.41961619],
       [0.01807649, 0.16069324, 0.82123027],
       [0.04516699, 0.14864374, 0.80618928],
       [0.01522213, 0.1210685 , 0.86370938],
       [0.13094334, 0.75222252, 0.11683415],
       [0.0831058 , 0.2822229 , 0.63467131],
       [0.03067143, 0.09108743, 0.87824114],
       [0.1805959 , 0.69091397, 0.12849013],
       [0.57921172, 0.26468461, 0.15610367],
       [0.